## Data Cleaning Process
This notebook covers the full cleaning pipeline for the UCI Online Retail Dataset including:
- Null value removal
- Cancellation separation
- Duplicate removal
- Feature engineering (Revenue, Month columns)

**Input:** online_retail.csv (541,910 rows)  
**Output:** online_retail_cleaned.csv (392,692 rows)

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("online_retail.csv")

In [3]:
report = pd.DataFrame(
    {
        "columns": df.columns.tolist(),
        "type": df.dtypes.values,
        "nulls": df.isnull().sum().values,
        "null_pct": (df.isnull().sum() / len(df) * 100.0).round(1).values,
        "unique": df.nunique().values,
        "duplicates_rows": df.duplicated().sum(),
    }
)
print(report.to_string())

       columns     type   nulls  null_pct  unique  duplicates_rows
0    InvoiceNo   object       0       0.0   25900             5268
1    StockCode   object       0       0.0    4070             5268
2  Description   object    1454       0.3    4223             5268
3     Quantity    int64       0       0.0     722             5268
4  InvoiceDate   object       0       0.0   23260             5268
5    UnitPrice  float64       0       0.0    1630             5268
6   CustomerID  float64  135080      24.9    4372             5268
7      Country   object       0       0.0      38             5268


In [4]:
text_cols = ["Description", "Country"]
for col in text_cols:
    df[col] = df[col].str.strip().str.title()

In [5]:
num_cols = ["Quantity", "UnitPrice"]
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [6]:
df = df[df["Quantity"] > 0]
df = df[df["UnitPrice"] > 0]

In [7]:
print(f" Null CustomerIDs:{df['CustomerID'].isnull().sum()}")
df.dropna(subset=["CustomerID"], inplace=True)
df["CustomerID"] = df["CustomerID"].astype(int)

 Null CustomerIDs:132220


In [8]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["Month"] = df["InvoiceDate"].dt.to_period("M")
df["Revenue"] = df["Quantity"] * df["UnitPrice"]

In [9]:
report = pd.DataFrame(
    {
        "columns": df.columns.tolist(),
        "type": df.dtypes.values,
        "nulls": df.isnull().sum().values,
        "null_pct": (df.isnull().sum() / len(df) * 100.0).round(1).values,
        "unique": df.nunique().values,
        "duplicates_rows": df.duplicated().sum(),
    }
)
print(report.to_string())

       columns            type  nulls  null_pct  unique  duplicates_rows
0    InvoiceNo          object      0       0.0   18532             5192
1    StockCode          object      0       0.0    3665             5192
2  Description          object      0       0.0    3866             5192
3     Quantity           int64      0       0.0     301             5192
4  InvoiceDate  datetime64[ns]      0       0.0   17282             5192
5    UnitPrice         float64      0       0.0     440             5192
6   CustomerID           int64      0       0.0    4338             5192
7      Country          object      0       0.0      37             5192
8        Month       period[M]      0       0.0      13             5192
9      Revenue         float64      0       0.0    2939             5192


In [10]:
dupes = df[df.duplicated()]
print(dupes.head(20))

    InvoiceNo StockCode                          Description  Quantity  \
517    536409     21866          Union Jack Flag Luggage Tag         1   
527    536409     22866        Hand Warmer Scotty Dog Design         1   
537    536409     22900       Set 2 Tea Towels I Love London         1   
539    536409     22111         Scottie Dog Hot Water Bottle         1   
555    536412     22327    Round Snack Boxes Set Of 4 Skulls         1   
587    536412     22273                 Feltcraft Doll Molly         1   
589    536412     22749    Feltcraft Princess Charlotte Doll         1   
594    536412     22141       Christmas Craft Tree Top Angel         1   
598    536412     21448            12 Daisy Pegs In Wood Box         1   
600    536412     22569          Feltcraft Cushion Butterfly         2   
601    536412     21448            12 Daisy Pegs In Wood Box         2   
604    536412     21448            12 Daisy Pegs In Wood Box         2   
605    536412     22902               

In [11]:
print(f"Rows before:{len(df)}")
df.drop_duplicates(inplace=True)

Rows before:397884


In [12]:
print(f"Rows after:{len(df)}")
print(f"Duplicates removed:{397884-len(df)}")

Rows after:392692
Duplicates removed:5192


In [13]:
invalid = [
    "Manual",
    "Postage",
    "Dotcom Postage",
    "Bank Charges",
    "Amazon Fee",
    "Cruk Commission",
]

In [14]:
df = df[~df["Description"].isin(invalid)]
print(df[df["Description"].isin(invalid)])

Empty DataFrame
Columns: [InvoiceNo, StockCode, Description, Quantity, InvoiceDate, UnitPrice, CustomerID, Country, Month, Revenue]
Index: []


In [15]:
report = pd.DataFrame(
    {
        "columns": df.columns.tolist(),
        "type": df.dtypes.values,
        "nulls": df.isnull().sum().values,
        "null_pct": (df.isnull().sum() / len(df) * 100.0).round(1).values,
        "unique": df.nunique().values,
        "duplicates_rows": df.duplicated().sum(),
    }
)
print(report.to_string())

       columns            type  nulls  null_pct  unique  duplicates_rows
0    InvoiceNo          object      0       0.0   18405                0
1    StockCode          object      0       0.0    3661                0
2  Description          object      0       0.0    3862                0
3     Quantity           int64      0       0.0     299                0
4  InvoiceDate  datetime64[ns]      0       0.0   17169                0
5    UnitPrice         float64      0       0.0     358                0
6   CustomerID           int64      0       0.0    4334                0
7      Country          object      0       0.0      37                0
8        Month       period[M]      0       0.0      13                0
9      Revenue         float64      0       0.0    2856                0


In [16]:
df.to_csv("online_retail_cleaned.csv", index=False)
print(f"File saved. Shape: {df.shape}")

File saved. Shape: (391286, 10)


In [17]:
df_raw = pd.read_csv("online_retail.csv", encoding="ISO-8859-1")

cancelled = df_raw[df_raw["InvoiceNo"].astype(str).str.startswith("C")].copy()
cancelled["Description"] = cancelled["Description"].str.strip().str.title()
cancelled["Quantity"] = pd.to_numeric(cancelled["Quantity"], errors="coerce")

invalid = [
    "Manual",
    "Postage",
    "Dotcom Postage",
    "Bank Charges",
    "Amazon Fee",
    "Cruk Commission",
]
cancelled = cancelled[~cancelled["Description"].isin(invalid)]
cancelled = cancelled[cancelled["Quantity"] > -10000]

cancelled.to_csv("online_retail_cancelled.csv", index=False)
print(f"Cancelled file saved! Shape: {cancelled.shape}")

Cancelled file saved! Shape: (8842, 8)


In [18]:
print(cancelled[cancelled["Description"] == "Manual"])
print(cancelled[cancelled["Quantity"] < -10000])
print(cancelled["Quantity"].min())

Empty DataFrame
Columns: [InvoiceNo, StockCode, Description, Quantity, InvoiceDate, UnitPrice, CustomerID, Country]
Index: []
Empty DataFrame
Columns: [InvoiceNo, StockCode, Description, Quantity, InvoiceDate, UnitPrice, CustomerID, Country]
Index: []
-9360
